In [2]:
import torch
from txtai import Embeddings
import pandas as pd
import numpy as np
#from datasets import load_dataset
print(torch.__version__)
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))
import bm25s
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from nltk.translate.meteor_score import meteor_score
from rouge_score import rouge_scorer
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from nltk.tokenize import word_tokenize
import nltk
from FlagEmbedding import FlagReranker
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from typing import List, Dict


2.14.1+cu126
True
NVIDIA GeForce RTX 4080 Laptop GPU


<h1>Define Functions</h1>

In [3]:
# Load ST model once for efficiency
#_st_model = SentenceTransformer("all-MiniLM-L6-v2")


def get_dense_embeddings(context_ids, context_docs):
    documents = list(zip(context_ids, context_docs))

    dense_embeddings = Embeddings(path="neuml/pubmedbert-base-embeddings", content=True)
    dense_embeddings.index(documents)
    return dense_embeddings

def get_sparse_embeddings(context_ids, context_docs):
    contexts_train_tokens = bm25s.tokenize(context_docs, stopwords="en")
    sparse_embeddings = bm25s.BM25(corpus=np.array(context_ids))    
    sparse_embeddings.index(contexts_train_tokens)
    return sparse_embeddings

def reciprocal_rank_fusion(ranks: List[List[str]], k: int = 60) -> List[str]:
    """
    ranks: list of lists of document IDs from each retrieval method
    k: constant for rank discount
    """
    doc_scores = {doc: 0.0 for doc in set().union(*ranks)}
    for rank_list in ranks:
        for rank, doc in enumerate(rank_list, start=1):
            doc_scores[doc] += 1 / (rank + k)
    return sorted(doc_scores.keys(), key=lambda d: doc_scores[d], reverse=True)

# Example usage:
# list1 = ["docA", "docB", "docC"]
# list2 = ["docB", "docC", "docD"]
# final = reciprocal_rank_fusion([list1, list2], k=60)
# print(final)

def retrieve_and_fuse(query,dense_embeddings, sparse_embeddings,corpus, dense_limit=25, sparse_limit=25, rrf_rank_discount=60):
    dense_list= dense_embeddings.search(query, limit=dense_limit)
    dense_ids = [dict['id'] for dict in dense_list]
    query_tokens = bm25s.tokenize(query, stopwords="en")
    sparse_ids, _ = sparse_embeddings.retrieve(query_tokens, k=sparse_limit)
    sparse_ids = sparse_ids[0].tolist()
    fused_ids = reciprocal_rank_fusion([dense_ids, sparse_ids], k=rrf_rank_discount)
    fused_results = [
    {
        "doc_id": int(doc_id),
        "text": corpus[int(doc_id)]
    }
    for doc_id in fused_ids
]

    return fused_results


def rerank_documents(query, fused_results, reranker):
    """
    Rerank documents using BGE cross encoder.

    Parameters
    ----------
    query : str
    fused_results : list[dict]
        Must contain keys:
            - doc_id
            - text

    Returns
    -------
    list[dict]
        Sorted by reranker score
    """

    query_doc_pairs = [
        [query, doc["text"]]
        for doc in fused_results
    ]

    scores = reranker.compute_score(
        query_doc_pairs,
        normalize=True
    )

    for doc, score in zip(fused_results, scores):
        doc["rerank_score"] = float(score)

    reranked = sorted(
        fused_results,
        key=lambda x: x["rerank_score"],
        reverse=True,
    )

    return reranked
def initialize_model_and_tokenizer(model_name):
    tokenizer = AutoTokenizer.from_pretrained(model_name)

    # Configure BitsAndBytes for 4-bit quantization
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16,
    )

    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        quantization_config=bnb_config,
        device_map="auto",
    )
    return model, tokenizer

def structure_message(query, top_docs):
    """
    Structures a message for the model to answer a biomedical question using the provided context.

    Args:
        query (str): The biomedical question.
        top_docs (list[str]): The top documents providing context. The list is expected to be ordered by relevance and 

    Returns:
        list[dict]: A list of messages structured for the model.
    """
    context = "\n\n".join(top_docs)
    messages = [
    {
        "role": "system",
        "content": (
            "You are a biomedical question answering assistant. "
            "Answer using only the provided context."
        )
    },
    {
    "role": "user",
    "content": f"""
        Question:
        {query}

        Context:
        {context}
        """
    }
    ]
    return messages

def serve_model(model, tokenizer, query, top_docs):
    message = structure_message(query, top_docs)
    text = tokenizer.apply_chat_template(message,tokenize=False,add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    outputs = model.generate(**inputs, max_new_tokens=150, temperature=0.1,)
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return response

def evaluate(reference_answer: str, candidate_answer: str, _st_model: SentenceTransformer) -> dict:
    """
    Computes BLEU, ROUGE, METEOR, and Semantic Text Similarity.

    Args:
        reference_answer (str): Ground-truth answer.
        candidate_answer (str): Model-generated answer.
        _st_model (SentenceTransformer): Preloaded SentenceTransformer model for semantic similarity.

    Returns:
        dict: Evaluation metrics.
    """

    # --------------------
    # BLEU
    # --------------------
    reference_tokens = [word_tokenize(reference_answer.lower())]
    candidate_tokens = word_tokenize(candidate_answer.lower())

    bleu = sentence_bleu(
        reference_tokens,
        candidate_tokens,
        smoothing_function=SmoothingFunction().method1,
    )

    # --------------------
    # ROUGE
    # --------------------
    rouge = rouge_scorer.RougeScorer(
        ["rouge1", "rouge2", "rougeL"],
        use_stemmer=True,
    )

    rouge_scores = rouge.score(reference_answer, candidate_answer)

    # --------------------
    # METEOR
    # --------------------
    meteor = meteor_score(
        [word_tokenize(reference_answer)],
        word_tokenize(candidate_answer),
    )

    # --------------------
    # Semantic Similarity (ST)
    # --------------------
    ref_emb = _st_model.encode([reference_answer])
    cand_emb = _st_model.encode([candidate_answer])

    semantic_similarity = cosine_similarity(ref_emb, cand_emb)[0][0]

    return {
        "BLEU": round(bleu, 4),
        "ROUGE-1": round(rouge_scores["rouge1"].fmeasure, 4),
        "ROUGE-2": round(rouge_scores["rouge2"].fmeasure, 4),
        "ROUGE-L": round(rouge_scores["rougeL"].fmeasure, 4),
        "METEOR": round(meteor, 4),
        "ST_SIMILARITY": round(float(semantic_similarity), 4),
    }

<h1>Prepare Data</h1>


In [6]:
source_data = pd.read_json('pubmedqa/data/pqaa_train_set.json').T
source_data['id'] = source_data.index.astype(int)
train = source_data.iloc[0:700]
test = source_data.iloc[700:1000]

In [ ]:
# Initialize embeddings and reranker models

QUERY_ID = 16097998
QUERY = train.loc[QUERY_ID, 'QUESTION']
RANK_THRESHOLD=0.5 # ALL RERANKED DOCUMENTS WITH RERANK SCORE ABOVE THIS THRESHOLD WILL BE KEPT
DECODER_MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct" # decoder model for generating response
CORPUS = {
    idx: " ".join(contexts)
    for idx, contexts in zip(
        train.index,
        train["CONTEXTS"]
    )
}
TRUE_RESPONSE = train.loc[QUERY_ID, 'LONG_ANSWER']

_st_model = SentenceTransformer("all-MiniLM-L6-v2") # _st_model for semantic similarity

#reranker is for ranking layer
reranker = FlagReranker(
    "BAAI/bge-reranker-v2-m3",
    use_fp16=True
)



contexts_docs_train = [" ".join(contexts) for contexts in train["CONTEXTS"]]
context_ids_train = train.index.astype(str).tolist()
#documents_train = list(zip(context_ids_train, contexts_docs_train))

contexts_docs_test = [" ".join(contexts) for contexts in test["CONTEXTS"]]
context_ids_test = test.index.astype(str).tolist()
#documents_test = list(zip(context_ids_test, contexts_docs_test))


print('train length:', len(train))
print('test length:', len(test))


dense_embeddings = get_dense_embeddings(context_ids_train, contexts_docs_train)
sparse_embeddings = get_sparse_embeddings(context_ids_train, contexts_docs_train)


Loading weights: 100%|██████████| 393/393 [00:00<00:00, 1823.05it/s]


train length: 700
test length: 300


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2656.59it/s]


<h1>Driver Function</h1>

In [8]:
def driver(query, dense_embeddings, sparse_embeddings, corpus, context_ids, context_docs,  _st_model=None, reranker=None,dense_limit=25, sparse_limit=25):

    if dense_embeddings is None:
        dense_embeddings = get_dense_embeddings(context_ids, context_docs)
    if sparse_embeddings is None:
        sparse_embeddings = get_sparse_embeddings(context_ids, context_docs)
    if _st_model is None:
        _st_model = SentenceTransformer("all-MiniLM-L6-v2")
    if reranker is None:
        reranker = FlagReranker("BAAI/bge-reranker-v2-m3", use_fp16=True)
        


    fused_results = retrieve_and_fuse(query, dense_embeddings, sparse_embeddings, corpus, dense_limit=dense_limit, sparse_limit=sparse_limit, rrf_rank_discount = 60)
    
    # Ranking layer
    
    top_k = rerank_documents(
        query=query,
        fused_results=fused_results,
        reranker=reranker
    )
    # filtration for lowly ranked documents
    top_k = [candidate_ref for candidate_ref in top_k if candidate_ref['rerank_score'] > RANK_THRESHOLD]
    #top_k_ids = [candidate_ref['doc_id'] for candidate_ref in top_k]
    top_k_text = [candidate_ref['text'] for candidate_ref in top_k]
    #top_k_rerank_scores = [candidate_ref['rerank_score'] for candidate_ref in top_k]


    # BEGIN GENERATION

    # get decoder model and tokenizer
    generator_model, generator_tokenizer = initialize_model_and_tokenizer(DECODER_MODEL_NAME)

    output = serve_model(generator_model, generator_tokenizer, query, top_k_text)

    candidate_response = output.split('\nassistant\n')[-1]


    score_dict = evaluate(candidate_response, query, _st_model)
    
    return score_dict, candidate_response, top_k

score_dict, candidate_response, top_k = driver(
    query=QUERY,
    dense_embeddings=dense_embeddings,
    sparse_embeddings=sparse_embeddings,
    corpus=CORPUS,
    context_ids=context_ids_train,
    context_docs=contexts_docs_train,
    _st_model=_st_model,
    reranker=reranker,
    dense_limit=25,
    sparse_limit=25
)

Loading weights: 100%|██████████| 339/339 [00:35<00:00,  9.66it/s]


In [15]:
print("QUESTION:\n", QUERY)
print('CANDIDATE RESPONSE:\n', candidate_response)
print('TRUE RESPONSE:\n', TRUE_RESPONSE)
print()
print(score_dict)

QUESTION:
 Is coeliac disease screening in risk groups justified?
CANDIDATE RESPONSE:
 Based on the provided context, coeliac disease screening in risk groups may be justified as it leads to high dietary compliance with a strict gluten-free diet, similar quality of life, and bone mineral density similar to the general population. This suggests that screening can effectively identify coeliac disease and improve outcomes when followed up with appropriate dietary management.
TRUE RESPONSE:
 Long-term dietary compliance in screen-detected patients was good. Quality of life and bone mineral density were comparable with those in non-coeliac subjects and the general population. Active screening in coeliac disease risk groups seems to be reasonable rather than harmful.

{'BLEU': 0.0015, 'ROUGE-1': 0.209, 'ROUGE-2': 0.1538, 'ROUGE-L': 0.209, 'METEOR': 0.1219, 'ST_SIMILARITY': 0.8282}
